# Training

Runs the 2x2 ablation as 5-fold cross-validation: 4 configs x 5 folds = 20 runs,
one per cell. About 35-60 minutes each, 14.3 GB of checkpoints in total.

Sections 4 and 5 read the archived records and need no GPU, no data and no
weights, so they work on a fresh clone.

Background on the guards and the traps is in `notebooks/README.md`.

## 1 · Setup

In [ ]:
import json
import os
import re
import subprocess
import sys

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
sys.path.insert(0, os.path.join(REPO, "src", "models"))

import numpy as np
import pandas as pd
from run_kfold import CONFIGS, N_FOLDS

LOG = os.path.join(REPO, "experiments_log")
for name, flags in CONFIGS.items():
    print(f"{name:16} {' '.join(flags)}")

## 2 · The runner

Each training cell calls `run_kfold.py` for a single fold, so the notebook and the
terminal take the same path and get the same guards: it refuses to start if the
disk is short, skips folds already finished, self-checks and archives afterwards.

Re-running a cell is safe. Epoch lines are thinned to keep the saved output small.

In [ ]:
STEP = os.path.join("src", "models", "run_kfold.py")
_epoch = re.compile(r"^(Epoch \d+/|\d+/\d+ - )")

def train(config, folds=None, every=25):
    """Run a config's five folds. `folds=[0]` re-runs just one.

    Finished folds are skipped, so re-running a cell after an interruption
    picks up where it stopped. Epoch lines are thinned to keep the saved
    notebook small; the full log is under experiments/kfold_logs/.
    """
    cmd = [sys.executable, STEP, config]
    if folds is not None:
        cmd += ["--folds", *map(str, folds)]
    print(" ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    n = 0
    for line in p.stdout:
        if _epoch.match(line):
            n += 1
            if n % every:
                continue
        print(line, end="", flush=True)
    p.wait()
    assert p.returncode == 0, f"exited {p.returncode} -- see experiments/kfold_logs/"

## 3 · The runs

One cell per config, five folds each, about 3 hours per cell. Run them in order.

Warning: while these are running, do not edit `src/models/` or `data/`. Each fold
is a fresh subprocess, so an edit lands on the next one and the five stop being
one experiment.

In [ ]:
train("cd_only")        # CD only

In [ ]:
train("lr_fix_only")        # CD + DCD

In [ ]:
train("rep_w05")        # CD + DCD + repulsion

In [ ]:
train("cd_rep05_full")        # CD + repulsion

## 4 · Were the runs clean?

One hard condition: a run is quotable only if EarlyStopping stopped it. Hitting
the epoch ceiling means it was still improving when cut off.

Three advisory ones, which the driver logs but does not stop on: enough LR decay
to have annealed, a settled tail, a val/train ratio that is not drifting.

In [ ]:
HARD = "EarlyStopping"
SOFT = {"lr_drops": (">=", 5), "tail30_std": ("<", 0.02), "val/train": ("<", 1.25)}

rows = []
for cfg in CONFIGS:
    for f in range(N_FOLDS):
        run = f"{cfg}_f{f}"
        meta = json.load(open(os.path.join(LOG, run, "run.json")))
        h = pd.read_csv(os.path.join(LOG, run, "history.csv"))
        n, best = len(h), int(h["val_loss"].idxmin()) + 1

        if n - best == meta["early_stop_patience"]:
            stop = HARD
        elif n >= meta["epochs"]:
            stop = "CEILING -- discard"
        else:
            stop = "wall clock"

        rows.append({"run": run, "epochs": n, "best": best, "stop": stop,
                     "lr_drops": int((h["lr"].diff() < -1e-12).sum()),
                     "tail30_std": round(float((h["val_cd_t_metric"] * meta["scale_mm"]).tail(30).std()), 4),
                     "val/train": round(float(h["val_cd_t_metric"][best - 1] / h["cd_t_metric"][best - 1]), 3),
                     "CD_t_mm": round(meta["best_val_cd_t_mm"], 3)})

runs = pd.DataFrame(rows)
print(runs.to_string(index=False))

assert (runs["stop"] == HARD).all(), runs[runs["stop"] != HARD]
print(f"\nHard condition: {len(runs)}/{len(runs)} stopped on EarlyStopping.")
for col, (op, lim) in SOFT.items():
    bad = runs[runs[col] >= lim] if op == "<" else runs[runs[col] < lim]
    print(f"  {col:12} {op} {lim:<6} " +
          ("clear" if bad.empty else ", ".join(f"{r.run} ({r[col]})" for _, r in bad.iterrows())))

`rep_w05_f0` trips the tail check at 0.039. It is kept: excluding it would move
the "+repulsion, with DCD" edge from -0.052 to -0.065 and halve its spread, i.e.
make this project's own claim look stronger.

`CD_t_mm` here is the training-time figure from `run.json` — best epoch, dataset
-average scale. It reads about 0.09 mm below the per-skull figure quoted from
`eval_all_runs.csv`. Use it to judge a run, never to compare configs.

In [ ]:
summary = runs.assign(config=runs.run.str.rsplit("_f", n=1).str[0])
print(summary.groupby("config", sort=False).agg(
    epochs_min=("epochs", "min"), epochs_max=("epochs", "max"),
    CD_t_mean=("CD_t_mm", "mean"), CD_t_std=("CD_t_mm", "std")).round(3).to_string())

## 5 · Curves

Five folds per config, so the spread between folds stays visible.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=len(CONFIGS), shared_yaxes=True,
                    subplot_titles=list(CONFIGS))
for col, cfg in enumerate(CONFIGS, start=1):
    for f in range(N_FOLDS):
        meta = json.load(open(os.path.join(LOG, f"{cfg}_f{f}", "run.json")))
        h = pd.read_csv(os.path.join(LOG, f"{cfg}_f{f}", "history.csv"))
        fig.add_scatter(x=np.arange(1, len(h) + 1), y=h["val_cd_t_metric"] * meta["scale_mm"],
                        name=f"fold {f}", legendgroup=f"f{f}", showlegend=(col == 1),
                        line=dict(width=1), row=1, col=col)

fig.update_yaxes(title_text="val CD_t (mm)", range=[6.0, 9.0], row=1, col=1)
fig.update_xaxes(title_text="epoch")
fig.update_layout(height=380, margin=dict(l=60, r=20, b=50, t=50)).show()